In [1]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
df = pd.read_csv("../data/LaptopPrices.csv")

In [4]:
def parse_memory(s):
    """'256GB SSD + 1TB HDD' -> toplam, SSD toplamı, HDD toplamı (GB cinsinden)"""
    total = ssd = hdd = 0.0
    for part in s.split("+"):
        m = re.search(r"(\d+\.?\d*)(GB|TB)", part)
        size = float(m.group(1)) * (1024 if m.group(2) == "TB" else 1)
        total += size
        if "SSD" in part or "Flash" in part:
            ssd += size
        elif "HDD" in part:
            hdd += size
        # Hybrid: sadece toplama dahil edilir
    return total, ssd, hdd

feat = pd.DataFrame(index=df.index)

In [5]:
feat["Company"]   = df["Company"]
feat["TypeName"]  = df["TypeName"]
feat["OpSys"]     = df["OpSys"]
feat["Cpu_brand"] = df["Cpu"].str.split().str[0]
feat["Gpu_brand"] = df["Gpu"].str.split().str[0]

In [6]:
feat["Inches"] = df["Inches"]
feat["Weight"] = df["Weight"].str.replace("kg", "", regex=False).astype(float)
feat["Ram"]    = df["Ram"].str.replace("GB", "", regex=False).astype(int)
feat["Cpu_ghz"] = df["Cpu"].str.extract(r"(\d+\.?\d*)GHz")[0].astype(float)

In [7]:
res = df["ScreenResolution"].str.extract(r"(\d+)x(\d+)").astype(float)
feat["ResW"] = res[0]
feat["ResH"] = res[1]
feat["PPI"]  = np.sqrt(feat["ResW"]**2 + feat["ResH"]**2) / feat["Inches"]
feat["IPS"]   = df["ScreenResolution"].str.contains("IPS").astype(int)
feat["Touch"] = df["ScreenResolution"].str.contains("Touchscreen").astype(int)

In [8]:
mem = df["Memory"].apply(parse_memory)
feat["Storage_total"] = [m[0] for m in mem]
feat["SSD_GB"]        = [m[1] for m in mem]
feat["HDD_GB"]        = [m[2] for m in mem]

In [9]:
assert feat.isna().sum().sum() == 0, "Eksik değer var, parser'ı kontrol et"

In [10]:
y = df["Price_euros"].values

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    feat, y, test_size=0.2, random_state=SEED
)

In [12]:
y_train_log = np.log1p(y_train)

In [13]:
cat_cols = ["Company", "TypeName", "OpSys", "Cpu_brand", "Gpu_brand"]
num_cols = [c for c in feat.columns if c not in cat_cols]

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ],
    sparse_threshold=0,  # Keras için yoğun (dense) matris
)

In [15]:
X_train_p = preprocessor.fit_transform(X_train).astype(np.float32)
X_test_p  = preprocessor.transform(X_test).astype(np.float32)

In [16]:
n_features = X_train_p.shape[1]
print("Özellik sayısı:", n_features)

Özellik sayısı: 53


In [17]:
model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1)  # regresyon: lineer çıkış
])

In [18]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │           6,912 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 15,233 (59.50 KB)

 Trainable params: 15,233 (59.50 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=30, restore_best_weights=True
)

In [21]:
history = model.fit(
    X_train_p, y_train_log,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)
print("Durdurulan epoch:", len(history.history["loss"]))

Durdurulan epoch: 121


In [22]:
pred_log = model.predict(X_test_p, verbose=0).ravel()
y_pred = np.expm1(pred_log)

In [23]:
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

In [24]:
baseline_mae = mean_absolute_error(y_test, np.full_like(y_test, np.median(y_train)))

In [25]:
model.save("../model/laptop_price.keras")